# Optimize the 3D Grating Coupler

Now that we know the 3D design and gradients look good, we can optimize the structure. Here we will preform non-stochastic optimization using the parameters from the 2D non-stochastic optimization as the starting parameters. 

In [1]:
import autograd.numpy as np 
import matplotlib.pyplot as plt

import tidy3d as td
import tidy3d.web as web
td.config.logging_level = "ERROR"

import pickle

from main import (make_sim, get_coupling_efficiency, projection_builder, run_adam,
                    R, r0_extra, initial_fill_factor, grating_period, 
                    etch_depth, to_substrate, N_teeth, n_wl, wl_range,
                    to_substrate)


14:12:53 EDT WARNING: Using canonical configuration directory at                
             '/Users/dominic/.config/tidy3d'. Found legacy directory at         
             '~/.tidy3d', which will be ignored. Remove it manually or run      
             'tidy3d config migrate --delete-legacy' to clean up.               

In [ ]:
# define the load function to get the initial parameters
def load_history(filename):
    with open(filename, "rb") as f:
        file = pickle.load(f)
    return file

# define the objective function
def objective(params,projection=lambda x: x):

    # project the parameters
    params_proj = projection(params)

    # unpack the projected parameters
    widths = params_proj[:-3]
    r0 = params_proj[-3]
    etch_depth = params_proj[-2]
    to_substrate = params_proj[-1]
    
    # run the simulation
    sim = make_sim(widths, r0=r0, etch_depth=etch_depth, to_substrate=to_substrate, include_field_monitor=False)
    sim_data = web.run(sim, task_name="GC4um_3D_initial_opt", verbose=False, path="data/tidy3d_output/tmp.hdf5")
    coupling = get_coupling_efficiency(sim_data)
    return np.mean(coupling)

Now we can load the previous result and run the optimization. 

In [ ]:
# load the previous parameters
data = load_history("../GC_4um_2D/data/opt/history.pkl")
params = data['history']['params'][-1]

# make the grating structure
widths = params[:-3]
r0 = params[-3] + r0_extra # need to add the taper length
etch_depth = params[-2]
to_substrate = params[-3]

# re-pack the parameters in an autograd array 
params = np.concatenate([widths, [r0], [etch_depth], [to_substrate]])

# define the projection for enforcing constraints
project, inverse_project = projection_builder()

# run the optimization
history, opt_state = run_adam(params, project, inverse_project, objective, num_steps=50, learning_rate=0.01, verbose=True)

step = 1
	J = 4.435e-01
	grad_norm = 6.8530e-01
